In [1]:
import sys
import os

# 1. Force install the correct libraries into the CURRENT kernel
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install xgboost pandas scikit-learn azure-storage-blob certifi joblib

# 2. Fix the macOS SSL issue for Azure downloads
import certifi
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
os.environ['SSL_CERT_FILE'] = certifi.where()

print("Setup complete. Environment is now configured.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.1 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Setup complete. Environment is now configured.


In [2]:
import platform
import struct
import xgboost as xgb

print(f"Architecture: {platform.architecture()[0]}")
print(f"Python bits: {struct.calcsize('P') * 8}")
print(f"XGBoost version: {xgb.__version__}")

Architecture: 64bit
Python bits: 64
XGBoost version: 3.2.0


### XGBoost Model ###

In [ ]:
import pandas as pd
import io
from azure.storage.blob import BlobServiceClient
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# --- DATA DOWNLOAD ---
blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
blob_client = blob_service_client.get_container_client("processed-data").get_blob_client("chicago_crime_features_v1.0.csv")

print("Downloading data...")
data = blob_client.download_blob().readall()
df = pd.read_csv(io.BytesIO(data))

# --- MODELING ---
# Using a 500k sample first to ensure it doesn't crash while we test
df_sample = df.sample(n=500000, random_state=42)
features = ['hour_sin', 'hour_cos', 'is_weekend', 'is_night', 'Latitude', 'Longitude', 'community_area_enc']

X = df_sample[features]
y = df_sample['crime_type_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Advanced XGBoost Settings
adv_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=10,
    learning_rate=0.1,
    tree_method="hist",  # Essential for speed on large data
    random_state=42
)

print("Training XGBoost...")
adv_model.fit(X_train, y_train)

# --- RESULTS ---
y_pred = adv_model.predict(X_test)
print(f"\nNew Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))

Training XGBoost...

New Accuracy: 28.87%
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       164
           1       0.11      0.01      0.02      6401
           2       0.25      0.50      0.34     18537
           3       0.20      0.06      0.09      5625
           4       0.00      0.00      0.00         9
           5       0.09      0.00      0.01       324
           6       0.19      0.10      0.13     11503
           7       0.00      0.00      0.00        54
           8       0.31      0.08      0.13      2758
           9       0.25      0.07      0.11      4083
          10       0.00      0.00      0.00       207
          11       0.00      0.00      0.00       150
          12       0.00      0.00      0.00       270
          13       0.50      0.02      0.04        55
          14       0.00      0.00      0.00       105
          15       0.00      0.00      0.00       187
          16       0.11      0.01      

In [9]:
import joblib

# Saving the advanced model locally
# This creates the .pkl file your partner will need for registration
model_filename = "advanced_xgboost_model.pkl"
joblib.dump(adv_model, model_filename)

print(f"Model successfully saved as {model_filename}")

Model successfully saved as advanced_xgboost_model.pkl


### Logistic regression model ###

In [11]:
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Feature preparation (Ensure these match your XGBoost features)
features = ['hour_sin', 'hour_cos', 'is_weekend', 'is_night', 'Latitude', 'Longitude', 'community_area_enc']
X = df[features]
y = df['crime_type_label']

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Corrected Pipeline (Fixed the TypeError from your screenshot)
logreg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000)) # Removed multi_class argument
])

print("Training Logistic Regression...")
logreg_pipe.fit(X_train, y_train)

# 4. Save
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, logreg_pipe.predict(X_test)):.2%}")
joblib.dump(logreg_pipe, "baseline_logreg_model.pkl")

Training Logistic Regression...
Logistic Regression Accuracy: 25.51%


['baseline_logreg_model.pkl']

### Decision Tree Model ###

In [12]:
from sklearn.tree import DecisionTreeClassifier

# Setting a max_depth prevents the model from becoming too large to save easily
dt_model = DecisionTreeClassifier(
    max_depth=12, 
    random_state=42
)

print("Training Decision Tree...")
dt_model.fit(X_train, y_train)

# Save
print(f"Decision Tree Accuracy: {accuracy_score(y_test, dt_model.predict(X_test)):.2%}")
joblib.dump(dt_model, "decision_tree_model.pkl")

Training Decision Tree...
Decision Tree Accuracy: 27.39%


['decision_tree_model.pkl']

### Comparing metrics ###

In [13]:
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score

# 1. Gather predictions for all models
# (Assuming models are trained and X_test/y_test are defined)
results = {
    "Logistic Regression": logreg_pipe.predict(X_test),
    "Decision Tree": dt_model.predict(X_test),
    "XGBoost (Advanced)": adv_model.predict(X_test)
}

# 2. Calculate metrics for each
comparison_data = []

for model_name, predictions in results.items():
    comparison_data.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
        "Macro F1-Score": f1_score(y_test, predictions, average='macro'),
        "Weighted Precision": precision_score(y_test, predictions, average='weighted', zero_division=0)
    })

# 3. Display as a clean Table
metrics_df = pd.DataFrame(comparison_data)
print("--- MODEL COMPARISON TABLE ---")
print(metrics_df.to_string(index=False))

# 4. Optional: Save summary for your report
metrics_df.to_csv("model_comparison_metrics.csv", index=False)

--- MODEL COMPARISON TABLE ---
              Model  Accuracy  Balanced Accuracy  Macro F1-Score  Weighted Precision
Logistic Regression  0.255126           0.043219        0.027902            0.138638
      Decision Tree  0.273875           0.061402        0.045840            0.215204
 XGBoost (Advanced)  0.287555           0.091775        0.076846            0.237559
